# P-2026-11-22-1: Why does Vietnam's HS4 ECI rank fall when commodity chapters are dropped or corrected?

**Question** (from `proposals/2026-11-20.md` #1, repeated in `proposals/2026-11-21.md` #3; both in open PRs #16/#17): under every HS4 resource variant tested on 11-20 (dropping chapters) and 11-21 (rebasing RCA), Vietnam's 2019 HS4 rank falls from 58 to 71–85 of 133, even though only about 2% of its exports are resources. Which mechanism drives this? Is it the products it loses, the RCA threshold, or the re-estimated product complexities (PCI)? And which of its products are responsible? Vietnam is a lab report country (Spring 2026, peers Singapore, Malaysia, Philippines).

**Why tonight:** `QUEUE.md` Q01–Q16 are all taken on `main`, so under PROTOCOL §1.4 the question comes from `proposals/`. This is the only proposal raised twice that isn't another resource-inside-ECI variant (11-21 #4 asks Will to close that line), and it doesn't modify `lib/`. It diagnoses something we already saw rather than adding a variant.

- **Date:** 2026-11-22 (America/New_York), nightly run. ID `P-2026-11-22-1`.
- **Data:** `data/raw/2019–2023_mega.csv` → repo HS4 via `lib/research.repo_hs` (chapters 98/99 dropped, reporters ≥ US$1B, ISO3). Benchmarks only: `external-data/raw/atlas_hs92_country_year.csv` (Growth Lab ECI) and `external-data/raw/wdi_panel.csv` (GDPpc PPP constant). No downloads.
- **Years:** 2019 primary; 2020–2022 checks; 2023 shown but suspect (D1).
- **Library:** `willbetts/nightly/lib/eci.py`, `research.py` (imported, not modified). The O4 SVD, the variants H1/H2/H3/HR (11-20) and C3 (11-21) are re-implemented here identically.
- **Run time:** printed in the last code cell (about 5 s).

## TL;DR

- **Vietnam's fall is mechanical, not a sign that its complexity was overstated.** Dropping ch. 25–27 (H1) takes Vietnam from **58 → 80** of 133 in 2019. The removed products themselves cost only **3** ranks. The rest splits about evenly (Shapley average over both step orders) between **RCA re-basing (9 ranks)** and **PCI re-estimation (10 ranks)**. The same holds for H2/H3/HR/C3 (re-basing 8–10 ranks each; total falls 13–27).
- **Re-basing is a denominator effect that hits every low-resource country.** Dropping chapters D multiplies all of a country's RCAs by f = (1 − world share of D)/(1 − own share of D). For Vietnam, f = **0.891** (ch. 25–27 are 12.76% of world exports but 2.12% of Vietnam's), so any product with H0 RCA below **1.122** drops out. Vietnam loses **20 of 238** RCA products. Those 20 have a higher mean PCI than the ones it keeps (−0.099 vs −0.419), and they include **integrated circuits (8542, US$11.5B, PCI 1.776, RCA 1.106 → 0.986)**. Across countries, Spearman(f, % of products lost) = **−0.916**.
- **The PCI channel runs through the apparel cluster, not through China specifically.** Vietnam's basket-mean PCI falls by 0.169, against 0.058 for all products. Ch. 61–64 (apparel and footwear) make up 40 of its 218 products but **33.7%** of the fall. Its top apparel co-exporters (Cambodia, China, Panama, Sri Lanka, Pakistan, ...) lose a mean of **0.273** SD of ECI. Across all countries, the rank fall tracks the apparel export share (Spearman **0.752**) more closely than the resource share (**−0.526**). Cambodia (27 ranks) and Vietnam (22) are the two largest fallers of 133.
- **H0 is the rank that matches the Atlas.** In 2019 on the 132-country common sample, Vietnam is **57th** under H0, **58th** under ATLAS ECI, and **92nd** in log GDPpc. Every variant moves it to 70–84, toward its income rank and away from ATLAS, and the same holds in 2020–2022. This is the country-level version of the pattern found on 11-19, 11-20 and 11-21. The peer comparison also shifts: Singapore and Malaysia hold or improve, while Philippines (f = 0.904) moves from 36 to 35–41.
- **Skipping re-basing doesn't rescue the drop.** H1-M drops ch. 25–27 from the binary matrix but keeps the H0 RCAs. It roughly halves Vietnam's fall (2019: 69 instead of 79 on the common sample) and halves the ATLAS loss (−0.020 vs −0.048), but it still loses ATLAS agreement in all 5 years and fails rule A. **For the lab:** use H0 HS4 ECI for Vietnam as is. Vietnam's IC specialisation sits just above RCA = 1, so its binary-RCA basket is fragile to any change in the denominator.

## Why this matters for the lab

Agenda §0 link 3 (the greedy optimizer's target is sound) assumes that a country's ECI reflects *its own* capabilities. If a country's rank can move 22 places because of a change in **other countries'** commodity exports, the lab needs to know which effect is real. Vietnam is a report country, and the HS4 move (11-14, Q04) is the lab's planned upgrade. If someone later proposes a resource adjustment at HS4, the Vietnam report would change by 13–27 ranks, so the lab should know *why* before accepting it. This diagnosis also gives the general rule: any drop or re-basing of resource products penalizes low-resource manufacturing exporters through the RCA denominator.

## Method (pre-stated before decomposing)

**Primary object:** Vietnam's 2019 HS4 rank (of 133) under H0 vs each variant: H1 −ch25–27, H2 −25–27,70–71, H3 −25–27,70–82, HR −26,27,71 (11-20), and C3 rebased RCA (resource products removed from the RCA denominators only, 11-21). The settings match 11-20/11-21: RCA ≥ 1, sorted SVD, O4 orientation.

**Pre-stated decomposition:** since ECI is exactly monotone in the mean PCI of a country's RCA products (checked in A), the rank at each step is computed by applying that step to *all* countries. The steps are: removal of the dropped products → RCA re-basing (new baskets, old PCI) → PCI re-estimation (new baskets, new PCI). The reverse order is also computed, and the Shapley average of the two orders is reported, because the steps interact.

**Pre-stated hypotheses:** (a) *denominator*: the re-basing step is large, and the factor f explains which countries lose products; (b) *co-exporters* (11-20's "Chinese-dominated chapters" guess): Vietnam's PCI losses concentrate in products whose co-exporters lose ECI.

**Benchmarks:** Vietnam's rank in ATLAS ECI and log GDPpc on the common sample, 2019–2022, plus the three report peers.

**Secondary (pre-stated):** H1-M, dropping ch. 25–27 from M without re-basing RCA, scored on M1 (ρ with ATLAS) and M2 (ρ with lnGDPpc) with 11-20 rule A. All variants tried are reported.

## Setup

In [1]:
import sys, time, warnings
T0 = time.time()
sys.path.insert(0, '../../lib')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
import eci, research as R
warnings.filterwarnings('ignore')
pd.set_option('display.width', 170); pd.set_option('display.max_columns', 30); pd.set_option('display.precision', 3)
FIG = Path('figures'); FIG.mkdir(exist_ok=True)
YEARS = [2019, 2020, 2021, 2022, 2023]; CHK = [2019, 2020, 2021, 2022]
VN, PEERS = 'VNM', ['SGP', 'MYS', 'PHL']          # Spring 2026 Vietnam report peers
def ch(a, b): return [f'{c:02d}' for c in range(a, b + 1)]
RES_CH = ['26', '27', '71']
HVARS = {'H1 -25-27': ch(25, 27), 'H2 -25-27,70-71': ch(25, 27) + ch(70, 71),
         'H3 -25-27,70-82': ch(25, 27) + ch(70, 82), 'HR -26,27,71': RES_CH}

def svd_o4(M):
    """Sorted SVD of binary M, O4 orientation (PCI negative with ubiquity), as 11-19/20/21.
    Returns z-scored ECI and z-scored PCI with the same sign."""
    Mv = M.to_numpy(float); kc, kp = Mv.sum(1), Mv.sum(0)
    B = Mv / np.sqrt(kc[:, None] * kp[None, :])
    U, S, Vt = np.linalg.svd(B, full_matrices=False)
    k = U[:, 1] / np.sqrt(kc); q = Vt[1] / np.sqrt(kp)
    s = -np.sign(np.corrcoef(q, kp)[0, 1]); k, q = k * s, q * s
    return (pd.Series((k - k.mean()) / k.std(), M.index), pd.Series((q - q.mean()) / q.std(), M.columns))

def rca_rebased(X, res_cols):
    """11-21 C3: resource products removed from the RCA denominators only (they stay in M)."""
    nr = X.drop(columns=res_cols); tot = nr.sum(axis=1)
    X = X.loc[tot > 0]; tot = tot[tot > 0]
    return X.div(tot, axis=0) / (X.sum(axis=0) / nr.loc[X.index].to_numpy().sum())

def binar(Rm):
    M = (Rm >= 1).astype(float)                       # Atlas-style >= 1, as 11-20/11-21
    return M.loc[M.sum(axis=1) > 0, M.sum(axis=0) > 0]

def basket_mean(M, pci):
    """Mean PCI over each country's RCA>=1 products (only products present in pci)."""
    cols = [c for c in M.columns if c in pci.index]
    Mm = M[cols]; Mm = Mm.loc[Mm.sum(axis=1) > 0]
    return (Mm @ pci[cols]) / Mm.sum(axis=1)

def rk(s, c=VN):
    return int(s.rank(ascending=False)[c]) if c in s.index else np.nan

XH = {y: R.repo_hs(y, 4) for y in YEARS}
RES = {y: R.resource_share(XH[y]) for y in YEARS}
AT = R.atlas_cy(); ATL = {y: AT.query('year == @y').set_index('iso3').eci for y in YEARS}
LNY = {y: np.log(R.gdppc(y, 'ppp')).dropna() for y in YEARS}
print('lib: eci.py research.py | numpy', np.__version__, '| pandas', pd.__version__)
print('HS4 countries x products (base):', {y: XH[y].shape for y in YEARS})
print('Vietnam in HS4 matrix every year:', {y: VN in XH[y].index for y in YEARS})

lib: eci.py research.py | numpy 2.5.3 | pandas 3.0.5
HS4 countries x products (base): {2019: (133, 1224), 2020: (130, 1223), 2021: (136, 1223), 2022: (119, 1232), 2023: (131, 1230)}
Vietnam in HS4 matrix every year: {2019: True, 2020: True, 2021: True, 2022: True, 2023: True}


### A. Exactness check: ECI is a monotone function of the mean PCI of a country's RCA products

In [2]:
M0 = {y: binar(eci.rca(XH[y])) for y in YEARS}
E0, P0 = {}, {}
for y in YEARS:
    E0[y], P0[y] = svd_o4(M0[y])
rows = []
for y in YEARS:
    s = basket_mean(M0[y], P0[y])
    rows.append({'year': y, 'n': len(s), 'pearson(ECI, basket-mean PCI)': np.corrcoef(s, E0[y][s.index])[0, 1],
                 'ranks identical': bool((s.rank(ascending=False) == E0[y][s.index].rank(ascending=False)).all()),
                 'VNM rank (H0)': rk(E0[y]), 'VNM ECI z (H0)': E0[y][VN]})
print(pd.DataFrame(rows).round(6).to_string(index=False))

 year   n  pearson(ECI, basket-mean PCI)  ranks identical  VNM rank (H0)  VNM ECI z (H0)
 2019 133                            1.0             True             58           0.040
 2020 130                            1.0             True             57           0.112
 2021 136                            1.0             True             53           0.208
 2022 119                            1.0             True             57          -0.032
 2023 131                            1.0             True             52           0.219


**Takeaway (A):** in every year, the Pearson correlation between ECI and the mean PCI of the country's RCA products is 1.0, and all ranks are identical, so the step-by-step rank decomposition below is exact. Vietnam's H0 rank is 52–58 in 2019–2023.

### B. Vietnam's rank under every variant, 2019–2023

In [3]:
def variant_M(X, v):
    if v == 'C3 rebased RCA':
        rc = [c for c in X.columns if c[:2] in RES_CH]
        return binar(rca_rebased(X, rc))
    keep = [c for c in X.columns if c[:2] not in HVARS[v]]
    return binar(eci.rca(X[keep]))
VARS = list(HVARS) + ['C3 rebased RCA']
MV, EV, PV = {}, {}, {}
for y in YEARS:
    for v in VARS:
        MV[y, v] = variant_M(XH[y], v); EV[y, v], PV[y, v] = svd_o4(MV[y, v])
rows = []
for y in YEARS:
    r = {'year': y, 'n': len(E0[y]), 'H0 base': rk(E0[y])}
    for v in VARS:
        r[v] = rk(EV[y, v])
    rows.append(r)
RB = pd.DataFrame(rows); print(RB.to_string(index=False))
print('\nRank change vs H0 (positive = falls):')
print(RB.set_index('year')[VARS].sub(RB.set_index('year')['H0 base'], axis=0).to_string())

 year   n  H0 base  H1 -25-27  H2 -25-27,70-71  H3 -25-27,70-82  HR -26,27,71  C3 rebased RCA
 2019 133       58         80               83               85            77              71
 2020 130       57         68               72               78            66              64
 2021 136       53         70               75               78            70              67
 2022 119       57         67               68               71            66              64
 2023 131       52         64               66               67            63              58

Rank change vs H0 (positive = falls):
      H1 -25-27  H2 -25-27,70-71  H3 -25-27,70-82  HR -26,27,71  C3 rebased RCA
year                                                                           
2019         22               25               27            19              13
2020         11               15               21             9               7
2021         17               22               25            17              

**Takeaway (B):** Vietnam falls under every variant in every year: 2019 by 13 (C3) to 27 (H3) ranks, and 2020–2023 by 6–25. Adding chapters to the drop list (H1 → H2 → H3) makes it worse, and C3, which removes nothing, still costs 13.

### C. Pre-stated decomposition of Vietnam's 2019 fall (H0 → variant)

Because ECI is exactly monotone in the basket-mean PCI (Section A), every country's rank at each step can be computed by applying the same step to **all** countries:

- **S0** = H0 baskets, H0 PCI (= H0 rank)
- **S1 removal**: H0 baskets minus the dropped chapters, still H0 PCI
- **S2 RCA re-basing**: the variant's baskets (RCA recomputed without the dropped chapters), still H0 PCI
- **S3 PCI re-estimation**: the variant's baskets and the variant's PCI (= variant rank)

The reverse order (S1 → re-estimate PCI on H0-minus-dropped baskets → re-base) is shown as a path-dependence check. For C3 nothing is removed, so S1 = S0.

In [4]:
def decompose(y, v, c=VN):
    M, p0, pv = M0[y], P0[y], PV[y, v]
    drop = [] if v.startswith('C3') else HVARS[v]
    Ma = M[[k for k in M.columns if k[:2] not in drop]]
    s0 = basket_mean(M, p0); s1 = basket_mean(Ma, p0); s2 = basket_mean(MV[y, v], p0); s3 = basket_mean(MV[y, v], pv)
    s1r = basket_mean(Ma, pv)                                      # reverse path: PCI first
    out = {'year': y, 'variant': v, 'S0 H0': rk(s0, c), 'S1 removal': rk(s1, c), 'S2 RCA re-basing': rk(s2, c),
           'S3 PCI re-est (= variant)': rk(s3, c), 'rev: PCI first': rk(s1r, c)}
    out['d removal'] = out['S1 removal'] - out['S0 H0']
    out['d re-basing'] = out['S2 RCA re-basing'] - out['S1 removal']
    out['d PCI'] = out['S3 PCI re-est (= variant)'] - out['S2 RCA re-basing']
    out['rev d PCI'] = out['rev: PCI first'] - out['S1 removal']
    out['rev d re-basing'] = out['S3 PCI re-est (= variant)'] - out['rev: PCI first']
    out['Shapley re-basing'] = (out['d re-basing'] + out['rev d re-basing']) / 2   # mean over the two orders
    out['Shapley PCI'] = (out['d PCI'] + out['rev d PCI']) / 2
    return out
DEC = pd.DataFrame([decompose(2019, v) for v in VARS])
print(DEC.to_string(index=False))
print('\nShapley split of the 2019 fall (removal, re-basing, PCI; total):')
print(DEC.assign(total=DEC['S3 PCI re-est (= variant)'] - DEC['S0 H0'])[['variant', 'd removal', 'Shapley re-basing', 'Shapley PCI', 'total']].to_string(index=False))
assert all(DEC['S3 PCI re-est (= variant)'] == RB.set_index('year').loc[2019, VARS].values)
print('\nH1 decomposition in every year:')
DEC_Y = pd.DataFrame([decompose(y, 'H1 -25-27') for y in YEARS])
print(DEC_Y[['year', 'S0 H0', 'S3 PCI re-est (= variant)', 'd removal', 'd re-basing', 'd PCI', 'rev d PCI', 'rev d re-basing', 'Shapley re-basing', 'Shapley PCI']].to_string(index=False))

 year         variant  S0 H0  S1 removal  S2 RCA re-basing  S3 PCI re-est (= variant)  rev: PCI first  d removal  d re-basing  d PCI  rev d PCI  rev d re-basing  Shapley re-basing  Shapley PCI
 2019       H1 -25-27     58          61                74                         80              75          3           13      6         14                5                9.0         10.0
 2019 H2 -25-27,70-71     58          64                74                         83              77          6           10      9         13                6                8.0         11.0
 2019 H3 -25-27,70-82     58          65                76                         85              77          7           11      9         12                8                9.5         10.5
 2019    HR -26,27,71     58          62                73                         77              68          4           11      4          6                9               10.0          5.0
 2019  C3 rebased RCA     58       

 year  S0 H0  S3 PCI re-est (= variant)  d removal  d re-basing  d PCI  rev d PCI  rev d re-basing  Shapley re-basing  Shapley PCI
 2019     58                         80          3           13      6         14                5                9.0         10.0
 2020     57                         68          2            3      6          6                3                3.0          6.0
 2021     53                         70          3            8      6          6                8                8.0          6.0
 2022     57                         67          3            6      1          5                2                4.0          3.0
 2023     52                         64          5            5      2          3                4                4.5          2.5


**Takeaway (C):** for H1 in 2019, removal costs 3 ranks. The order of the other two steps matters (forward: re-basing 13, PCI 6; reverse: PCI 14, re-basing 5), so the Shapley split is the fair summary: re-basing 9.0 and PCI 10.0 of the 22-rank fall. Re-basing is 8.0–10.0 ranks for every variant. PCI ranges from 3.5 (C3) to 11.0 (H2). Over 2019–2023, H1 removal is always small (2–5), and the other two steps together give 7–19 ranks, with the 2019 fall the largest.

### D. Why re-basing costs Vietnam products: the denominator factor

Dropping chapters D from both the country total and the world total multiplies every surviving RCA of country c by the same factor
f_c = (1 − share of D in world exports) / (1 − share of D in c's exports).
A country with a below-world share of D gets f_c < 1, so every product with H0 RCA in [1, 1/f_c) falls below the threshold.

In [5]:
def factor(X, drop):
    cols = [k for k in X.columns if k[:2] in drop]
    wd = X[cols].to_numpy().sum() / X.to_numpy().sum(); cd = X[cols].sum(axis=1) / X.sum(axis=1)
    return (1 - wd) / (1 - cd)
y = 2019; X = XH[y]; drop = HVARS['H1 -25-27']
f = factor(X, drop)
keep = [k for k in X.columns if k[:2] not in drop]
R0 = eci.rca(X)[keep]; R1 = eci.rca(X[keep])
print('max |RCA_H1 / (f * RCA_H0) - 1| over positive cells:', float(np.nanmax(np.abs((R1 / R0.mul(f, axis=0)).to_numpy()[R0.to_numpy() > 0] - 1))))
world_share = X[[k for k in X.columns if k[:2] in drop]].to_numpy().sum() / X.to_numpy().sum()
print(f'2019 world share of ch. 25-27 in repo HS4 exports: {world_share:.4f}')
print(f"Vietnam: share of ch. 25-27 = {1 - (1 - world_share) / f[VN]:.4f}, f = {f[VN]:.4f}, threshold for keeping RCA>=1 becomes RCA_H0 >= {1 / f[VN]:.4f}")
lost = ((R0 >= 1) & (R1 < 1)).sum(axis=1); gained = ((R0 < 1) & (R1 >= 1)).sum(axis=1)
T = pd.DataFrame({'f': f, 'resource share (26/27/71)': RES[y], 'RCA products lost': lost, 'gained': gained,
                  'H0 diversity': (R0 >= 1).sum(axis=1)})
T['net lost %'] = (T['RCA products lost'] - T['gained']) / T['H0 diversity'] * 100
print('\nSpearman(f, net lost %):', round(stats.spearmanr(T.f, T['net lost %'])[0], 3), '| countries with f < 1:', int((T.f < 1).sum()), 'of', len(T))
print('\nVietnam, peers, and the 8 largest net losers (%):')
print(T.loc[[VN] + PEERS].round(3).to_string()); print(T.sort_values('net lost %', ascending=False).head(8).round(3).to_string())
p0 = P0[y]
vl = R0.loc[VN][(R0.loc[VN] >= 1) & (R1.loc[VN] < 1)]
L = pd.DataFrame({'RCA H0': vl, 'RCA H1': R1.loc[VN, vl.index], 'PCI H0 (z)': p0.reindex(vl.index),
                  'VNM exports $M': X.loc[VN, vl.index] / 1e6}).sort_values('PCI H0 (z)', ascending=False)
print(f'\nVietnam products lost by re-basing: {len(L)}; mean H0 PCI {L["PCI H0 (z)"].mean():.3f} vs mean H0 PCI of the products it keeps '
      f'{p0.reindex([k for k in M0[y].columns[M0[y].loc[VN] > 0] if k[:2] not in drop]).mean():.3f}')
print(L.round(3).to_string())

max |RCA_H1 / (f * RCA_H0) - 1| over positive cells: 2.886579864025407e-15
2019 world share of ch. 25-27 in repo HS4 exports: 0.1276
Vietnam: share of ch. 25-27 = 0.0212, f = 0.8913, threshold for keeping RCA>=1 becomes RCA_H0 >= 1.1220

Spearman(f, net lost %): -0.916 | countries with f < 1: 75 of 133

Vietnam, peers, and the 8 largest net losers (%):
         f  resource share (26/27/71)  RCA products lost  gained  H0 diversity  net lost %
VNM  0.891                      0.023                 20       0           238       8.403
SGP  1.003                      0.178                  0       0           156       0.000
MYS  1.030                      0.163                  0      10           229      -4.367
PHL  0.904                      0.058                 13       0           159       8.176
         f  resource share (26/27/71)  RCA products lost  gained  H0 diversity  net lost %
HKG  0.874                      0.102                 24       0           175      13.714
CHE  0.8

**Takeaway (D):** the identity is exact (max error 2.9e-15). Vietnam's f = 0.891 means a product needs H0 RCA ≥ 1.122 to survive. 20 products fail, none are gained, and the 20 have a higher mean PCI (−0.099) than Vietnam's remaining basket (−0.419). So losing them lowers its average. The costliest is integrated circuits (8542: US$11.5B, PCI 1.776, RCA 1.106 → 0.986), followed by 8505 magnets and 8504 transformers. 75 of 133 countries have f < 1, and f predicts the share of RCA products lost (Spearman −0.916). The peers differ: Philippines loses 13 products (f 0.904), Singapore none (f 1.003), and Malaysia gains 10 (f 1.030). Hong Kong, Switzerland, Czechia, France and Germany lose the largest shares (11–14%).

### E. Why PCI re-estimation also hurts Vietnam: its products' co-exporters lose ECI

In [6]:
y = 2019; v = 'H1 -25-27'; M1v, p1, e1 = MV[y, v], PV[y, v], EV[y, v]
dE = (e1 - E0[y]).dropna()
basket = M1v.columns[M1v.loc[VN] > 0]
dP = (p1[basket] - P0[y][basket])
print(f'Vietnam H1 basket: {len(basket)} products; mean dPCI {dP.mean():.3f}; mean dPCI over all H1 products {(p1 - P0[y].reindex(p1.index)).mean():.3f}')
# co-exporter mechanism: PCI_p is (affine in) the mean ECI of its exporters
co = M1v[basket]
mean_dE_exporters = (co.T @ dE.reindex(co.index).fillna(0)) / co.sum(axis=0)
print('Spearman(dPCI, mean dECI of the product\'s H1 exporters), Vietnam basket:', round(stats.spearmanr(dP, mean_dE_exporters[dP.index])[0], 3))
byc = pd.DataFrame({'dPCI': dP, 'ch': dP.index.str[:2]}).groupby('ch').dPCI.agg(['sum', 'count', 'mean'])
byc['share of total dPCI'] = byc['sum'] / dP.sum()
print('\nChapters contributing most to the fall in Vietnam\'s basket-mean PCI:')
print(byc.sort_values('sum').head(8).round(3).to_string())
g = [k for k in basket if k[:2] in ('61', '62', '63', '64')]
print(f'\nApparel/footwear (ch. 61-64): {len(g)} of {len(basket)} products, {dP[g].sum() / dP.sum():.3f} of the total dPCI')
# which countries co-export Vietnam's apparel/footwear and how did their ECI move?
cnt = M1v.loc[:, g].sum(axis=1).drop(VN).sort_values(ascending=False).head(12)
CO = pd.DataFrame({'shared ch.61-64 RCA products': cnt, 'dECI H0->H1': dE.reindex(cnt.index), 'resource share': RES[y].reindex(cnt.index),
                   'H0 rank': E0[y].rank(ascending=False).reindex(cnt.index), 'H1 rank': e1.rank(ascending=False).reindex(cnt.index)})
print('\nTop co-exporters of Vietnam\'s ch.61-64 products (H1):'); print(CO.round(3).to_string())
print(f"\nmean dECI: these 12 co-exporters {CO['dECI H0->H1'].mean():.3f} | all countries {dE.mean():.3f}")
print('\nVietnam products with the largest PCI falls:')
TOP = pd.DataFrame({'dPCI': dP, 'PCI H0': P0[y][dP.index], 'PCI H1': p1[dP.index], 'exporters H1': co.sum(axis=0)[dP.index],
                    'mean dECI of exporters': mean_dE_exporters[dP.index]}).sort_values('dPCI').head(12)
print(TOP.round(3).to_string())

Vietnam H1 basket: 218 products; mean dPCI -0.169; mean dPCI over all H1 products -0.058
Spearman(dPCI, mean dECI of the product's H1 exporters), Vietnam basket: 0.7

Chapters contributing most to the fall in Vietnam's basket-mean PCI:
      sum  count   mean  share of total dPCI
ch                                          
61 -5.122     15 -0.341                0.139
62 -4.390     14 -0.314                0.119
85 -2.043     12 -0.170                0.055
64 -1.583      6 -0.264                0.043
40 -1.388      5 -0.278                0.038
63 -1.338      5 -0.268                0.036
94 -1.175      4 -0.294                0.032
65 -1.132      4 -0.283                0.031

Apparel/footwear (ch. 61-64): 40 of 218 products, 0.337 of the total dPCI

Top co-exporters of Vietnam's ch.61-64 products (H1):
     shared ch.61-64 RCA products  dECI H0->H1  resource share  H0 rank  H1 rank
KHM                          36.0       -0.506           0.030     88.0    115.0
CHN                   

**Takeaway (E):** PCI falls for Vietnam's products (mean −0.169) about 3× more than for the average product (−0.058). A product's ΔPCI tracks the mean ΔECI of its exporters (Spearman 0.70). Apparel and footwear (ch. 61–64) contribute 33.7% of the fall with 40 of 218 products: knitwear 6104/6105/6108/6110, 6206, footwear 6404. Their main co-exporters are Cambodia, China, Myanmar, Panama, Indonesia, Tunisia, Sri Lanka and Pakistan, which mostly have low resource shares and lose a mean of 0.273 SD of ECI (all countries: −0.000). China does fall (18 → 27), but it is one of a broad low-resource apparel cluster, so the 11-20 "Chinese-dominated chapters" guess is only partly right. Electronics also lose PCI: 8529 (1.528 → 0.955) and 8517 (0.752 → 0.296).

### F. Who else falls like Vietnam? Rank change vs resource share and apparel share (2019, H1)

In [7]:
y = 2019; X = XH[y]
rank0 = E0[y].rank(ascending=False); rank1 = EV[y, 'H1 -25-27'].rank(ascending=False)
F = pd.DataFrame({'rank fall': rank1 - rank0, 'resource share': RES[y], 'f': factor(X, HVARS['H1 -25-27']),
                  'apparel share 61-64': X[[k for k in X.columns if k[:2] in ('61', '62', '63', '64')]].sum(axis=1) / X.sum(axis=1)})
for c in ['resource share', 'f', 'apparel share 61-64']:
    print(f'Spearman(rank fall, {c}) = {stats.spearmanr(F["rank fall"], F[c])[0]:.3f}')
low = F['resource share'] < 0.10
print(f"\nCountries with resource share < 0.10: n = {int(low.sum())}, mean rank fall {F.loc[low, 'rank fall'].mean():.1f}; "
      f"others: n = {int((~low).sum())}, mean {F.loc[~low, 'rank fall'].mean():.1f}")
print('\n10 largest fallers:'); print(F.sort_values('rank fall', ascending=False).head(10).round(3).to_string())
print(f"\nVietnam's fall ({int(F.loc[VN, 'rank fall'])}) is the #{int(F['rank fall'].rank(ascending=False, method='min')[VN])} largest of {len(F)}")
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.scatter(F['resource share'], F['rank fall'], s=14, color='#9a9994')
for c, col in [(VN, '#d6402a')] + [(p, '#2a78d6') for p in PEERS]:
    ax.scatter(F.loc[c, 'resource share'], F.loc[c, 'rank fall'], s=40, color=col, zorder=3)
    ax.annotate(c, (F.loc[c, 'resource share'], F.loc[c, 'rank fall']), xytext=(4, 3), textcoords='offset points', fontsize=8)
ax.axhline(0, color='#52514e', lw=0.8); ax.set_xlabel('2019 resource share (HS 26/27/71)'); ax.set_ylabel('HS4 rank fall, H0 -> H1 (+ = worse)')
for s_ in ['top', 'right']: ax.spines[s_].set_visible(False)
ax.set_title('Dropping ch. 25-27: low-resource countries fall, resource exporters rise', fontsize=9, loc='left')
fig.tight_layout(); fig.savefig(FIG / 'rank_fall_vs_resource.png', dpi=110); plt.close(fig); print('saved figures/rank_fall_vs_resource.png')

Spearman(rank fall, resource share) = -0.526
Spearman(rank fall, f) = -0.583
Spearman(rank fall, apparel share 61-64) = 0.752

Countries with resource share < 0.10: n = 53, mean rank fall 6.5; others: n = 80, mean -4.3

10 largest fallers:
     rank fall  resource share      f  apparel share 61-64
KHM       27.0           0.030  0.873                0.654
VNM       22.0           0.023  0.891                0.195
SLV       21.0           0.037  0.904                0.378
GTM       15.0           0.049  0.918                0.130
MUS       15.0           0.071  0.881                0.297
LKA       15.0           0.056  0.900                0.448
PAK       14.0           0.020  0.907                0.422
PSE       14.0           0.001  0.903                0.035
NIC       14.0           0.107  0.875                0.295
DOM       13.0           0.238  0.923                0.109

Vietnam's fall (22) is the #2 largest of 133


saved figures/rank_fall_vs_resource.png


![rank fall](figures/rank_fall_vs_resource.png)

**Takeaway (F):** the fall is systematic. Countries with resource share < 0.10 fall by 6.5 ranks on average (n = 53), and the others rise by 4.3 (n = 80). Rank fall tracks the apparel share (Spearman 0.752) more than the resource share (−0.526) or f (−0.583). The top fallers are Cambodia (27), Vietnam (22), El Salvador (21), Guatemala, Mauritius and Sri Lanka (15 each). They are all low-resource apparel or light-manufacturing exporters.

### G. Which rank is right for Vietnam? ATLAS and income benchmarks, plus the report peers

In [8]:
rows = []
for y in CHK:
    c = [k for k in E0[y].index if k in ATL[y].index and k in LNY[y].index]
    if VN not in c: continue
    r = {'year': y, 'n common': len(c), 'ATLAS rank': int(ATL[y][c].rank(ascending=False)[VN]),
         'lnGDPpc rank': int(LNY[y][c].rank(ascending=False)[VN]), 'H0': int(E0[y][c].rank(ascending=False)[VN])}
    for v in VARS:
        e = EV[y, v].reindex(c).dropna(); r[v] = int(e.rank(ascending=False)[VN])
    rows.append(r)
GB = pd.DataFrame(rows); print(GB.to_string(index=False))
print('\nPeer ranks (2019, all 133 HS4 countries):')
print(pd.DataFrame({'H0': {p: rk(E0[2019], p) for p in [VN] + PEERS}, **{v: {p: rk(EV[2019, v], p) for p in [VN] + PEERS} for v in VARS}}).to_string())

 year  n common  ATLAS rank  lnGDPpc rank  H0  H1 -25-27  H2 -25-27,70-71  H3 -25-27,70-82  HR -26,27,71  C3 rebased RCA
 2019       132          58            92  57         79               82               84            76              70
 2020       129          54            90  56         67               71               77            65              63
 2021       135          51            91  52         69               74               77            69              66
 2022       118          51            83  56         66               67               70            65              63

Peer ranks (2019, all 133 HS4 countries):


     H0  H1 -25-27  H2 -25-27,70-71  H3 -25-27,70-82  HR -26,27,71  C3 rebased RCA
VNM  58         80               83               85            77              71
SGP   5          5                2                2             2               5
MYS  23         19               19               19            19              19
PHL  36         40               40               41            35              37


**Takeaway (G):** H0 puts Vietnam within 1–5 ranks of its ATLAS rank in 2019–2022 (2019: 57 vs 58). Every variant moves it 9–26 ranks below ATLAS in 2019–2022 (2019: 70–84), toward its income rank (83–92). Within the report's peer group, the variants widen the gap below Singapore (5 → 2–5) and Malaysia (23 → 19), and Philippines moves to 35–41 (from 36).

### H. Secondary (pre-stated): does a drop that *skips the re-basing* avoid the problem?

**H1-M** drops chapters 25–27 from the **binary M** (columns of M0) but keeps the H0 RCAs, so there's no denominator effect. Scored like 11-20 (M1 = Spearman with ATLAS ECI, M2 = Spearman with log GDPpc), on the countries that have H0, ATLAS and GDPpc each year. Rule (same as 11-20 rule A): it beats H0 on both M1 and M2 in 2019 **and** in ≥ 2 of 2020–2022. This is a side check, not a recommendation to adopt it.

In [9]:
rows = []
EM = {}
for y in YEARS:
    Mm = M0[y][[k for k in M0[y].columns if k[:2] not in HVARS['H1 -25-27']]]; Mm = Mm.loc[Mm.sum(axis=1) > 0]
    EM[y], _ = svd_o4(Mm)
    c = [k for k in E0[y].index if k in ATL[y].index and k in LNY[y].index and k in EM[y].index and k in EV[y, 'H1 -25-27'].index]
    for name, e in [('H0 base', E0[y]), ('H1 -25-27', EV[y, 'H1 -25-27']), ('H1-M (no re-basing)', EM[y])]:
        rows.append({'year': y, 'variant': name, 'n': len(c), 'M1 rho ATLAS': stats.spearmanr(e[c], ATL[y][c])[0],
                     'M2 rho lnGDPpc': stats.spearmanr(e[c], LNY[y][c])[0], 'VNM rank': int(e[c].rank(ascending=False)[VN])})
HM = pd.DataFrame(rows)
for m in ['M1 rho ATLAS', 'M2 rho lnGDPpc']:
    HM['d' + m[:2]] = HM[m] - HM.groupby('year')[m].transform('first')
print(HM.round(3).to_string(index=False))
def ruleA(v):
    d = HM[HM.variant == v].set_index('year')
    win = (d.dM1 > 0) & (d.dM2 > 0)
    return bool(win[2019] and win[[2020, 2021, 2022]].sum() >= 2)
print('\nRule A pass:', {v: ruleA(v) for v in ['H1 -25-27', 'H1-M (no re-basing)']})
HM.round(4).to_csv('h1m_metrics.csv', index=False)
print(f'\nRun time: {time.time() - T0:.1f} s')

 year             variant   n  M1 rho ATLAS  M2 rho lnGDPpc  VNM rank    dM1   dM2
 2019             H0 base 132         0.972           0.781        57  0.000 0.000
 2019           H1 -25-27 132         0.923           0.837        79 -0.048 0.056
 2019 H1-M (no re-basing) 132         0.951           0.822        69 -0.020 0.040
 2020             H0 base 129         0.973           0.807        56  0.000 0.000
 2020           H1 -25-27 129         0.947           0.848        67 -0.026 0.041
 2020 H1-M (no re-basing) 129         0.949           0.838        62 -0.024 0.031
 2021             H0 base 135         0.965           0.768        52  0.000 0.000
 2021           H1 -25-27 135         0.919           0.829        69 -0.046 0.061
 2021 H1-M (no re-basing) 135         0.939           0.818        60 -0.026 0.050
 2022             H0 base 118         0.968           0.777        56  0.000 0.000
 2022           H1 -25-27 118         0.941           0.837        66 -0.027 0.060
 202

**Takeaway (H):** H1-M removes the denominator effect and about halves Vietnam's 2019 fall (common-sample rank 57 → 69, vs 79 for H1) and the 2019 ATLAS loss (ΔM1 −0.020 vs −0.048). It still raises income ρ less than H1 does, loses ATLAS agreement in all 5 years (−0.016 to −0.027), and fails rule A. So does H1. There's no variant here worth adopting.

## Caveats

- **One year, one country, primary.** The decomposition is done in full for 2019 (all variants) and for H1 in 2019–2023. The other years show the same direction, with smaller falls (10–17 ranks under H1, 2020–2023).
- **The decomposition depends on step order.** Re-basing and PCI re-estimation interact (forward vs reverse orders differ by 8 ranks in 2019), so the Shapley split is a fair summary, not a unique attribution.
- **Binary RCA amplifies everything here.** Vietnam's RCA in integrated circuits is 1.106, so a ~11% change in the denominator flips one of its highest-PCI products. A continuous RCA (Q07) would make this smooth. That's not tested tonight.
- **Nominal 2019 values, exporter-reported HS4, ≥ US$1B reporters.** World resource share (12.76% for ch. 25–27) depends on 2019 commodity prices, so the size of f changes with oil prices. The rank falls shrink in 2020 (an oil-price trough), which fits this.
- **Panama's** apparel co-exports likely reflect Colón Free Zone re-exports, not domestic production. Vietnam's own exports also include assembly with imported inputs (a known limit of gross-export ECI).
- **2023 is suspect (D1).** It's shown but not used for any conclusion. D4 (chapter mapping) doesn't apply at HS4.

## Helpful to know

- **ECI is exactly monotone in the mean PCI of a country's RCA products** (Pearson 1.0 in every year). Any rank change can be decomposed exactly by recomputing these basket means for all countries. This is a cheap diagnostic for any future variant.
- **Dropping product chapters D rescales every RCA in a row by one factor f_c = (1 − world share of D)/(1 − country share of D).** Low-resource countries get f < 1 and lose products near the threshold. Resource exporters get f > 1 and gain them. 11-21's C3 "rebased RCA" has the same effect without removing anything, which is why it behaved like deletion there.
- **Vietnam's 2019 integrated-circuit RCA is only 1.106.** Its biggest high-complexity product sits right at the binary threshold, so its HS4 ECI is sensitive to anything that moves the RCA denominator: data revisions, the HS revision mix, and re-export treatment.
- **Cambodia, not Vietnam, is the largest faller** (27 ranks). Any resource tweak at HS4 penalises the low-resource apparel cluster as a group.

## New research avenues (also in `proposals/2026-11-22.md`)

1. **Threshold fragility map for report countries.** For each Spring 2026 country (and peers), count RCA ≥ 1 products with RCA in [1, 1.15) and their PCI. Then compute how far the HS4 rank moves if those products are flipped one at a time (exact via basket means). Vietnam's 8542 at RCA 1.106 shows why this matters. One night.
2. **Continuous-RCA ECI for Vietnam's case** (Q07 at the country level). Under log(1+RCA) or capped RCA, does Vietnam's fall under H1/C3 shrink to about the removal-only size (3 ranks)? This tests whether binary RCA causes most of the sensitivity.
3. **Basket-mean decomposition as a standard tool.** Add a `rank_decomposition()` helper (removal / re-basing / PCI, both orders, Shapley) to a future `lib/` change, and re-explain 11-20's Canada rise (45 → 30) with it. Canada is a resource exporter with f > 1, so it's the mirror case.
4. **Year-to-year Vietnam movement in the reports.** The Spring 2026 Vietnam report quotes grouped ECI changes. Apply the same decomposition to Vietnam's HS4 2019 → 2022 change (fixed panel, as 11-15) to separate its own basket changes from the PCI drift caused by other countries.

## Data issues / data needed

None new. All data used is already in the repo or in `external-data/raw/`. Known issues that touch this result: D1 (2023 suspect, not used for conclusions) and nominal values (the world resource share follows commodity prices).